In [1]:
import re
import os 
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from pyhive import presto
from datetime import date
from collections import Counter

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'

conn=presto.connect('presto-gateway.processing.data.production.internal',presto_port,username)

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


pd.set_option('display.max_columns', None)

In [2]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        -- event_type,
        -- click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1
        )
    ORDER BY 1,2,3

"""

overall_entertainment=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62377/158498091.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_entertainment=pd.read_sql(overall_query,conn)


In [3]:
overall_entertainment

,test_group,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,59899,181010,123661,68.32,85.28,65,111,254,163,10,0.64,10.17,16.63,3.91,1156,18408,30104,7075
1,test,60083,181597,123740,68.14,85.32,72,114,259,167,10,0.73,10.19,16.86,3.75,1331,18499,30622,6817


In [6]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        event_type,
        click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3

"""

overall_entertainment=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62377/2026902862.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_entertainment=pd.read_sql(overall_query,conn)


In [10]:
overall_entertainment

,test_group,event_type,click_type,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,59899,181010,123661,68.32,85.28,65,110,254,163,10,0.64,10.17,16.63,3.91,1156,18408,30104,7075
1,test,viewed,close_click,45849,133546,90750,67.95,85.04,75,114,264,171,10,0.70,10.41,16.79,3.84,929,13902,22427,5134
2,test,viewed,play_click,2629,4793,2998,62.55,82.41,56,94,216,141,10,1.15,12.21,19.36,4.24,55,585,928,203
3,test,viewed,None,1948,3883,2655,68.37,87.41,39,108,199,151,9,0.75,7.67,18.72,4.17,29,298,727,162
4,test,None,None,16797,39434,27376,69.42,86.40,74,113,255,166,10,0.81,9.44,16.61,3.35,318,3721,6551,1320


In [17]:
overall_entertainment.to_clipboard(index=False)

## pre-post

In [12]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        -- event_type,
        -- click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1
        )
    ORDER BY 1,2,3

"""

overall_entertainmentpre=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62377/4259449240.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_entertainmentpre=pd.read_sql(overall_query,conn)


In [18]:
overall_entertainmentpre

,test_group,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,58334,187403,110872,59.16,75.39,54,120,265,158,11,0.64,16.96,15.92,7.00,1206,31787,29842,13115
1,test,58571,187428,111383,59.43,75.68,74,121,270,159,11,0.66,16.66,15.97,6.99,1243,31229,29932,13098


In [20]:
overall_entertainmentpre.to_clipboard(index=False)

## COBRA TTC 

In [25]:
overall_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        -- event_type,
        -- click_type,
        cobra_ttc_bucket,
        
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,

            CASE 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec <= 15 THEN 'a. 0-15 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 16 AND 30 THEN 'b. 16-30 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 31 AND 60 THEN 'c. 31-60 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 61 AND 180 THEN 'd. 61-180 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec > 180 THEN 'e. 180+ sec' 
            WHEN modified_order_status != 'COBRA' THEN 'f. NO COBRA'
            ELSE 'Check' END cobra_ttc_bucket,
            
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
           and click_type = 'play_click'
           
        GROUP BY 1,2
        )
    ORDER BY 1,2,3

"""

overall_entertainment_ttc=pd.read_sql(overall_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62377/1744123129.py:193: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_entertainment_ttc=pd.read_sql(overall_query,conn)


In [26]:
overall_entertainment_ttc

,test_group,cobra_ttc_bucket,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,test,a. 0-15 sec,56,58,0,0.00,0.00,NaN,12.0,NaN,12,NaN,0.00,100.0,0.00,0.00,0,58,0,0
1,test,b. 16-30 sec,73,76,0,0.00,0.00,NaN,21.0,NaN,21,NaN,0.00,100.0,0.00,0.00,0,76,0,0
2,test,c. 31-60 sec,73,81,0,0.00,0.00,NaN,45.0,NaN,45,NaN,0.00,100.0,0.00,0.00,0,81,0,0
3,test,d. 61-180 sec,194,251,0,0.00,0.00,NaN,114.0,NaN,114,NaN,0.00,100.0,0.00,0.00,0,251,0,0
4,test,e. 180+ sec,102,119,0,0.00,0.00,NaN,235.0,NaN,235,NaN,0.00,100.0,0.00,0.00,0,119,0,0
5,test,f. NO COBRA,2515,4208,2998,71.25,93.87,56.0,NaN,216.0,201,10.0,1.31,0.0,22.05,4.82,55,0,928,203


In [27]:
overall_entertainment_ttc.to_clipboard(index=False)

## Other

In [24]:
overall_sports.to_csv('overall_entertainment.csv',index=False)

In [5]:
overall_daywise_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        yyyymmdd,
        test_group,
        event_type,
        click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            yyyymmdd,
            test_group,
            event_type,
            click_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

overall_daywise_entertainment=pd.read_sql(overall_daywise_query,conn)   

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62377/274808176.py:181: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_daywise_entertainment=pd.read_sql(overall_daywise_query,conn)


In [38]:
overall_daywise_entertainment

,yyyymmdd,test_group,event_type,click_type,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,20240911,control,None,None,18235,31529,21527,68.28,84.95,67,109,253,163,10,0.60,10.19,16.35,4.25,190,3213,5155,1341
1,20240911,test,viewed,close_click,9895,18199,11950,65.66,83.13,84,110,256,161,10,0.99,11.14,17.18,4.72,180,2028,3127,859
2,20240911,test,viewed,play_click,406,782,498,63.68,82.10,91,98,193,136,11,0.64,12.66,17.52,4.60,5,99,137,36
3,20240911,test,viewed,None,321,555,373,67.21,87.93,18,107,181,142,8,0.90,7.21,20.36,3.96,5,40,113,22
4,20240911,test,None,None,7677,12216,8639,70.72,87.12,69,112,272,174,10,0.63,8.94,16.04,3.30,77,1092,1960,403
5,20240912,control,None,None,17992,30804,21302,69.15,85.51,56,109,235,155,10,0.68,9.82,16.00,3.99,211,3025,4930,1228
6,20240912,test,viewed,close_click,13575,23601,16384,69.42,85.25,81,113,252,161,10,0.70,10.05,15.57,3.99,166,2372,3674,942
7,20240912,test,viewed,play_click,530,938,571,60.87,79.64,50,103,200,135,10,2.45,12.47,18.34,5.44,23,117,172,51
8,20240912,test,viewed,None,422,756,497,65.74,84.66,25,90,203,137,8,1.32,7.80,18.78,6.22,10,59,142,47
9,20240912,test,None,None,3557,5716,4002,70.01,86.44,60,110,232,149,10,1.03,9.45,16.08,3.08,59,540,919,176


In [39]:
overall_daywise_entertainment.to_clipboard(index=False)

In [27]:
overall_daywise_entertainment.to_csv('overall_daywise_entertainment.csv',index=False)

In [41]:
overall_session_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        
        test_group,
        event_type,
        click_type,
        time_period,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
           
            test_group,
            event_type,
            click_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

overall_session_entertainment=pd.read_sql(overall_session_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_52024/4137295872.py:189: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_session_entertainment=pd.read_sql(overall_session_query,conn)


In [42]:
overall_session_entertainment

,test_group,event_type,click_type,time_period,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,afternoon,32665,64634,46978,72.68,89.28,61.0,106,260,175,9,0.44,7.43,16.29,2.85,285,4802,10529,1840
1,control,None,None,evening,26411,50312,33069,65.73,83.61,54.0,112,266,169,12,0.85,10.74,17.53,4.78,430,5404,8819,2407
2,control,None,None,morning,17618,33889,23001,67.87,82.04,61.0,112,232,147,10,0.61,12.50,13.90,4.85,206,4237,4712,1642
3,control,None,None,rest_evening,10896,17865,10999,61.57,81.32,93.0,111,265,163,12,0.65,13.84,19.24,4.15,117,2472,3437,742
4,control,None,None,rest_morning,8861,14264,9582,67.18,85.62,88.0,112,205,149,8,0.83,10.45,18.21,3.11,118,1490,2597,443
5,test,viewed,close_click,afternoon,24662,47931,34692,72.38,88.77,63.0,113,267,179,9,0.41,7.90,16.13,2.92,198,3788,7729,1398
6,test,viewed,close_click,evening,20105,37637,24638,65.46,83.80,60.0,110,272,172,12,0.86,10.68,18.05,4.64,324,4021,6794,1746
7,test,viewed,close_click,morning,13292,24887,16754,67.32,81.97,55.0,117,249,155,10,0.61,13.03,14.41,4.39,153,3242,3586,1093
8,test,viewed,close_click,rest_evening,8106,13201,8033,60.85,80.99,51.0,117,270,172,12,0.67,13.84,19.64,4.45,89,1827,2593,588
9,test,viewed,close_click,rest_morning,6220,9875,6622,67.06,84.84,124.0,120,227,157,7,1.67,10.36,17.44,3.13,165,1023,1722,309


In [43]:
overall_session_entertainment.to_clipboard(index=False)

In [30]:
overall_session_entertainment.to_csv('overall_session_entertainment.csv',index=False)

In [45]:
overall_cluster_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            rr_net.cluster_name,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        event_type,
        click_type,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            cluster_name,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

"""

overall_cluster_entertainment=pd.read_sql(overall_cluster_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_52024/3318123340.py:198: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_cluster_entertainment=pd.read_sql(overall_cluster_query,conn)


In [47]:
overall_cluster_entertainment=overall_cluster_entertainment[overall_cluster_entertainment['click_type']=='play_click']

In [48]:
overall_cluster_entertainment

,test_group,event_type,click_type,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
24,test,viewed,play_click,Banjara Hills,59,86,52,60.47,79.07,NaN,95,214,191,23,0.00,15.12,18.60,5.81,0,13,16,5
25,test,viewed,play_click,Bodduppal,27,35,27,77.14,94.29,NaN,209,176,176,6,0.00,2.86,17.14,2.86,0,1,6,1
26,test,viewed,play_click,Gandipet,20,26,16,61.54,73.08,NaN,16,208,105,7,0.00,15.38,11.54,11.54,0,4,3,3
27,test,viewed,play_click,Hitech City,154,231,114,49.35,72.73,58.0,64,137,109,11,0.87,18.61,23.38,7.79,2,43,54,18
28,test,viewed,play_click,Kothaguda 2,41,64,34,53.13,75.00,151.0,114,226,184,9,1.56,12.50,21.88,10.94,1,8,14,7
29,test,viewed,play_click,Kukatpally 3,64,88,58,65.91,88.64,11.0,138,187,138,11,1.14,5.68,21.59,4.55,1,5,19,4
30,test,viewed,play_click,Manikonda,50,75,45,60.00,82.67,NaN,185,272,233,9,0.00,12.00,22.67,5.33,0,9,17,4
31,test,viewed,play_click,Marredpally,54,65,48,73.85,89.23,NaN,115,186,136,12,0.00,10.77,15.38,0.00,0,7,10,0
32,test,viewed,play_click,Mehedipatnam,77,118,74,62.71,78.81,NaN,73,488,111,10,0.00,17.80,15.25,3.39,0,21,18,4
33,test,viewed,play_click,Mothinagar,44,66,41,62.12,77.27,11.0,119,189,119,11,1.52,15.15,15.15,6.06,1,10,10,4


In [49]:
overall_cluster_entertainment.to_clipboard(index=False)

In [52]:
overall_cluster_query_benchmark=f"""


with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
)




SELECT
        
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            rr_net
        where 
            yyyymmdd is not null
           
        GROUP BY 1
        )
    ORDER BY 1

"""

overall_cluster_entertainment_benchmark=pd.read_sql(overall_cluster_query_benchmark,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_52024/1331892875.py:128: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_cluster_entertainment_benchmark=pd.read_sql(overall_cluster_query_benchmark,conn)


In [53]:
overall_cluster_entertainment_benchmark

,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,Banjara Hills,31257,58019,36406,62.75,81.58,60,114,293,175,15,1.08,12.40,18.47,4.92,628,7196,10718,2856
1,Bodduppal,20126,35946,28865,80.30,94.50,55,102,243,182,7,0.28,3.72,13.69,1.50,100,1338,4922,539
2,Gandipet,6944,13920,7712,55.40,74.95,103,125,273,167,10,1.65,16.81,19.15,6.57,230,2340,2665,914
3,Hitech City,57528,116724,72058,61.73,79.29,61,111,238,152,12,1.00,14.25,17.26,5.45,1167,16637,20147,6356
4,Kothaguda 2,18078,34797,22162,63.69,81.25,87,116,267,163,11,0.76,13.05,17.21,4.93,263,4541,5987,1716
5,Kukatpally 3,28348,49167,36279,73.79,90.13,30,104,252,173,9,0.47,6.54,15.90,2.85,230,3217,7818,1401
6,Manikonda,17866,34212,23986,70.11,87.28,35,112,254,170,10,0.27,9.17,16.84,3.26,93,3137,5760,1116
7,Marredpally,17377,29046,21097,72.63,89.32,57,108,278,181,10,0.22,8.20,16.29,2.24,64,2382,4731,651
8,Mehedipatnam,31442,58502,40223,68.75,86.41,58,109,278,177,11,0.88,8.60,17.30,4.09,516,5033,10120,2393
9,Mothinagar,16857,32854,21412,65.17,83.35,97,115,282,177,11,0.89,11.08,17.87,4.68,291,3641,5871,1536


In [54]:
overall_cluster_entertainment_benchmark.to_clipboard(index=False)

In [55]:
overall_cluster_session_query=f"""
with exp_category as(
select
    userid as user_id,
    case    when name='CaptainSearchEntertainmentNewsTest' then 'test' 
            else 'control'
    end as test_group
from
   canonical.user_selector
where
    yyyymmdd = '20240905'
    and selectorid in ('66d9ebe38a94b100011bfea8','66d9ed054c9b3100015a374a')

),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
),

view_tbl as(

    select
        yyyymmdd,
        profile_identity as user_id,
        'viewed' as event_type
    from 
        clevertap.customer_notification_viewed_immutable 
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),

click_tbl as(
    select
        yyyymmdd,
        profile_identity as user_id,
        CASE
                    when event_props_wzrk_c2a='Button 2' then 'play_click'
                    else 'close_click'
        end as click_type
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_campaign_id = '1725971599'

),
merge as(
    select exp_category.*,
            rr_net.yyyymmdd,
            rr_net.order_id,
            rr_net.order_status,
            rr_net.modified_order_status,
            rr_net.spd_fraud_flag,
            rr_net.quarter_hour,
            rr_net.time_spent_in_sec,
            rr_net.time_taken_to_accp,
            rr_net.cluster_name,
            view_tbl.event_type,
            click_tbl.click_type
    from
        exp_category
    left join 
            rr_net 
        on exp_category.user_id=rr_net.customer_id
    left join
            view_tbl
        on rr_net.customer_id=view_tbl.user_id
           and rr_net.yyyymmdd=view_tbl.yyyymmdd
    left join
            click_tbl
        on rr_net.customer_id=click_tbl.user_id
        and view_tbl.yyyymmdd=click_tbl.yyyymmdd

)

SELECT
        test_group,
        event_type,
        click_type,
        time_period,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4,5
        )
    ORDER BY 1,2,3,4,5

"""

overall_cluster_entertainment_session=pd.read_sql(overall_cluster_session_query,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_52024/2913115380.py:206: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_cluster_entertainment_session=pd.read_sql(overall_cluster_session_query,conn)


In [56]:
overall_cluster_entertainment_session=overall_cluster_entertainment_session[overall_cluster_entertainment_session['click_type']=='play_click']

In [57]:
overall_cluster_entertainment_session

,test_group,event_type,click_type,time_period,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
120,test,viewed,play_click,afternoon,Banjara Hills,19,26,14,53.85,92.31,NaN,185.0,407.0,311.0,11.0,0.00,3.85,38.46,3.85,0,1,10,1
121,test,viewed,play_click,afternoon,Bodduppal,8,9,6,66.67,88.89,NaN,209.0,2229.0,209.0,6.0,0.00,11.11,22.22,0.00,0,1,2,0
122,test,viewed,play_click,afternoon,Gandipet,10,12,8,66.67,83.33,NaN,11.0,1016.0,105.0,8.0,0.00,8.33,16.67,8.33,0,1,2,1
123,test,viewed,play_click,afternoon,Hitech City,56,61,47,77.05,95.08,15.0,27.0,97.0,47.0,8.0,1.64,3.28,18.03,0.00,1,2,11,0
124,test,viewed,play_click,afternoon,Kothaguda 2,17,24,16,66.67,91.67,NaN,110.0,173.0,144.0,9.0,0.00,8.33,25.00,0.00,0,2,6,0
125,test,viewed,play_click,afternoon,Kukatpally 3,24,33,22,66.67,96.97,NaN,138.0,412.0,412.0,8.0,0.00,3.03,27.27,0.00,0,1,9,0
126,test,viewed,play_click,afternoon,Manikonda,17,24,15,62.50,87.50,NaN,173.0,397.0,173.0,8.0,0.00,4.17,25.00,8.33,0,1,6,2
127,test,viewed,play_click,afternoon,Marredpally,17,18,15,83.33,94.44,NaN,15.0,186.0,186.0,14.0,0.00,5.56,11.11,0.00,0,1,2,0
128,test,viewed,play_click,afternoon,Mehedipatnam,34,55,32,58.18,74.55,NaN,73.0,488.0,82.0,10.0,0.00,18.18,16.36,7.27,0,10,9,4
129,test,viewed,play_click,afternoon,Mothinagar,11,14,11,78.57,92.86,NaN,NaN,55.0,55.0,10.0,0.00,0.00,14.29,7.14,0,0,2,1


In [58]:
overall_cluster_entertainment_session.to_clipboard(index=False)

In [59]:
overall_cluster_session_query_benchmark=f"""


with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Hitech City', 
                'Shamshabad', 
                'Kukatpally 3', 
                'Bodduppal', 
                'Mehedipatnam', 
                'Kothaguda 2', 
                'Mothinagar', 
                'Banjara Hills', 
                'Gandipet', 
                'Manikonda',
                'Marredpally'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        AND customer_id in (
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Hyderabad'
            and platform='Android'
            and app_version>='8.30.0'           
            )
         
)




SELECT
        time_period,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            rr_net
        where 
            yyyymmdd is not null
           
        GROUP BY 1,2
        )
    ORDER BY 1,2

"""

overall_cluster_session__entertainment_benchmark=pd.read_sql(overall_cluster_session_query_benchmark,conn)   

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_52024/2006519071.py:134: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_cluster_session__entertainment_benchmark=pd.read_sql(overall_cluster_session_query_benchmark,conn)


In [60]:
overall_cluster_session__entertainment_benchmark

,time_period,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,afternoon,Banjara Hills,14197,20861,14647,70.21,88.19,52,114,288,187,12,0.50,8.57,17.69,2.74,104,1788,3690,572
1,afternoon,Bodduppal,9295,13336,11068,82.99,96.21,70,107,255,202,6,0.10,2.64,12.73,1.06,13,352,1698,141
2,afternoon,Gandipet,3134,5029,2972,59.10,78.25,87,124,261,163,8,1.39,15.15,18.85,5.21,70,762,948,262
3,afternoon,Hitech City,22798,33435,25382,75.91,92.82,54,90,225,166,8,0.51,4.97,16.58,1.70,170,1661,5542,568
4,afternoon,Kothaguda 2,7589,11000,8231,74.83,91.52,94,103,262,182,9,0.53,5.65,16.40,2.30,58,622,1804,253
5,afternoon,Kukatpally 3,12133,17273,13413,77.65,93.47,54,98,265,187,8,0.30,4.45,15.40,1.79,51,768,2660,309
6,afternoon,Manikonda,7773,11670,8744,74.93,91.37,22,102,250,174,9,0.24,6.02,16.22,2.37,28,703,1893,276
7,afternoon,Marredpally,8138,11646,8502,73.00,89.33,60,106,299,189,11,0.20,7.90,15.99,2.57,23,920,1862,299
8,afternoon,Mehedipatnam,14807,23165,16469,71.09,88.60,36,106,283,184,11,0.63,7.34,17.18,3.43,145,1701,3979,794
9,afternoon,Mothinagar,7080,10540,7707,73.12,90.87,73,108,282,192,10,0.81,6.12,17.46,2.20,85,645,1840,232


In [61]:
overall_cluster_session__entertainment_benchmark.to_clipboard(index=False)

In [15]:
click_bucket=f"""
with olf as (

    SELECT
        yyyymmdd,
        customer_id,
        order_id,
        order_requested_epoch,
        customer_cancelled_epoch,
        dropped_epoch,
        expired_epoch,
        rider_accepted_epoch
        
    FROM 
        orders.order_logs_fact
    WHERE
        yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Hyderabad'
        
    ),




click as(
    select
        yyyymmdd,
        profile_identity as user_id,
        epoch as click_epoch
    from 
        clevertap.customer_notification_clicked_immutable
    where yyyymmdd >= '20240911' and yyyymmdd<='20240916'
    and event_props_wzrk_c2a='Button 2'
    and event_props_campaign_id = '1725971599'

),


merge AS (
    
    SELECT 
        olf.*,
        click_epoch,
        (click_epoch / 1000 - order_requested_epoch / 1000) time_to_click
        --row_number() over(partition by olf.yyyymmdd,olf.customer_id,olf.order_id order by click_epoch) seq_no
    FROM 
        olf
    JOIN 
        click
        on olf.yyyymmdd = click.yyyymmdd
        and olf.customer_id = click.user_id
        and olf.order_requested_epoch < click.click_epoch
        and (olf.customer_cancelled_epoch > click.click_epoch or dropped_epoch > click.click_epoch or 
        expired_epoch > click.click_epoch or rider_accepted_epoch> click.click_epoch)
    )
    
    
   
    
    
    SELECT
        
        case 
        when time_to_click <= 15 then 'a. 15 Sec'
        when time_to_click between 16 and 30 then 'b. 16-30 Sec'
        when time_to_click between 31 and 60 then 'c. 30-60 Sec'
        when time_to_click between 61 and 120 then 'd. 1-2 min'
        when time_to_click between 121 and 240 then 'e. 3-4 min'
        when time_to_click > 240 then 'f. 4+ min' 
        else 'check' end as click_bucket,
        count(distinct customer_id) customers
    FROM 
        merge
    -- WHERE 
    --     --seq_no = 1
    GROUP BY 1
    ORDER BY 1

"""
click_bucket_df = pd.read_sql(click_bucket,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62377/198338273.py:82: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_bucket_df = pd.read_sql(click_bucket,conn)


In [16]:
click_bucket_df

,click_bucket,customers
0,a. 15 Sec,2036
1,b. 16-30 Sec,405
2,c. 30-60 Sec,146
3,d. 1-2 min,76
4,e. 3-4 min,53
5,f. 4+ min,32


In [21]:
click_bucket_df.customers.sum()

2748

In [22]:
2629 - 2748

-119

In [23]:
2036-119


1917

In [ ]:
click_bucket_df

In [64]:
total_customers = click_bucket_df['customers'].sum()

# Calculate the percentage of customers for each click_bucket
click_bucket_df['customer_percentage'] = (click_bucket_df['customers'] / total_customers) * 100

# Round the percentage to 2 decimal places
click_bucket_df['customer_percentage'] = click_bucket_df['customer_percentage'].round(2)


In [24]:
click_bucket_df.to_clipboard(index=False)